In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [2]:
## Load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [3]:
# Load the pre-trained model with ReLU activation
model = load_model('simple_rnn_imdb.h5')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [14]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
# Same values used during training
max_features = 10000
max_len = 500

# Load dataset
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

# Pad sequences
X_test = pad_sequences(X_test, maxlen=max_len)

In [15]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)

print("Loss:", loss)
print("Accuracy:", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - accuracy: 0.8438 - loss: 0.3703
Loss: 0.37026235461235046
Accuracy: 0.8438400030136108


In [16]:
print("good:", word_index.get("good"))
print("great:", word_index.get("great"))
print("acting:", word_index.get("acting"))
print("movie:", word_index.get("movie"))
print("engaging:", word_index.get("engaging"))

good: 49
great: 84
acting: 113
movie: 17
engaging: 1725


In [17]:
tests = [
    "good",
    "bad",
    "great",
    "terrible",
    "excellent",
    "awful",
    "I loved this movie",
    "I hated this movie",
    "This movie was fantastic",
    "This movie was awful"
]

for t in tests:
    sentiment, score = predict_sentiment(t)
    print(f"{score:.4f} | {sentiment:8} | {t}")

['good']
[52]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
0.4911 | Negative | good
['bad']
[78]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
0.3490 | Negative | bad
['great']
[87]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
0.5848 | Positive | great
['terrible']
[394]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
0.1314 | Negative | terrible
['excellent']
[321]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
0.9462 | Positive | excellent
['awful']
[373]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
0.0267 | Negative | awful
['i', 'loved', 'this', 'movie']
[13, 447, 14, 20]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
0.8597 | Positive | I loved this movie
['i', 'hated', 'this', 'movie']
[13, 1800, 14, 20]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
0.3544 | Negative | I hated this movie
['this', 'movie', 'was', 'fantastic']
[14, 20, 16, 777]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
0.5338 | Positive | This movie was fantastic
['this', 'movie', 'was', 'awful']
[14, 20, 16, 373]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
0.0178 | Negative | This movie 

In [4]:
model.get_weights()

[array([[ 0.10286303,  0.24120396, -0.05790769, ..., -0.12408271,
          0.16703941, -0.03040539],
        [-0.1175084 , -0.07468797,  0.01891268, ...,  0.01981537,
          0.20443027, -0.07089826],
        [-0.06650247, -0.00615266,  0.09970942, ..., -0.1323871 ,
         -0.01556988, -0.10256989],
        ...,
        [ 0.10550935,  0.04109226, -0.07320299, ..., -0.08695292,
         -0.06099744,  0.08192436],
        [-0.02145474, -0.04792014,  0.06193012, ..., -0.00584349,
          0.01442497,  0.00522588],
        [ 0.05770012, -0.00477899, -0.06144355, ...,  0.00778758,
         -0.00451228,  0.03705189]], shape=(10000, 128), dtype=float32),
 array([[ 1.4376107e-01, -1.1400275e-01,  3.0576847e-02, ...,
         -1.5433395e-01, -5.4321898e-04,  1.0609018e-01],
        [ 5.1529448e-02,  1.2794320e-01, -8.6140387e-02, ...,
         -7.3946282e-02,  7.2264783e-02, -4.4972934e-05],
        [-6.2884078e-03,  2.1162361e-02,  9.6662693e-02, ...,
          2.6859783e-02, -1.4326598e

In [5]:
# Step-2: Padded functions
# Function to decode reviews
def decode_review(encoded_review):
    return ''.join([reverse_word_index.get(i-3,'?') for i in encoded_review])

# function to preprocess user input
from tensorflow.keras.preprocessing.text import text_to_word_sequence

def preprocess_text(text):
    words = text_to_word_sequence(text)

    encoded = [word_index.get(word, 2) + 3 for word in words]

    return sequence.pad_sequences([encoded], maxlen=500)

In [9]:
### prediction function

def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)

    words = text_to_word_sequence(review)
    print(words)

    encoded = [word_index.get(word, 2) + 3 for word in words]
    print(encoded)

    prediction = model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]

In [34]:
# Step 4: User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."
 
sentiment, score = predict_sentiment(example_review)

['this', 'movie', 'was', 'fantastic', 'the', 'acting', 'was', 'great', 'and', 'the', 'plot', 'was', 'thrilling']
[14, 20, 16, 777, 4, 116, 16, 87, 5, 4, 114, 16, 3017]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


In [35]:
print(f'Review:{example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

Review:This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Negative
Prediction Score: 0.47865432500839233
